# TimeLapse Cleaning — Denoising & Robustness Experiments

Playground for trying different processing techniques on the **noisy migrated
B-scans** (space domain) and the **cross-spectrum** (phase-plane domain) used
for phase-plane shift estimation in `TimeLapse_Processing.ipynb` Sections 2b/2c.

Two independent places to clean the data, kept as separate sections so techniques
from each can be mixed and matched in the benchmark at the bottom:

- **B-scan-domain cleaning** (Section 3): operates on the full migrated image,
  before cropping/apex-finding.
- **Cross-spectrum-domain cleaning** (Section 4): operates on the 2-D FFT
  cross-spectrum of the (already cropped) base/monitor pair, before the
  shift estimate is extracted from it.

Everything here was validated against the real noisy Kirchhoff dataset during
development (see dev notes / conversation history); the headline finding was
that the crop-centering step (apex-finding) — not the fit method — was the
actual cause of most small-shift failures. That fix (Section 2) is included
here so every cross-spectrum-domain candidate gets a fair, correctly-centred
crop to work with.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert as sp_hilbert, medfilt2d
from scipy.signal.windows import tukey
from scipy.ndimage import uniform_filter, gaussian_filter
from skimage.restoration import denoise_wavelet
from sklearn.linear_model import HuberRegressor, RANSACRegressor, LinearRegression

In [ ]:
# ── Standard figure auto-save (protocol: .wiki/FIGURES_PROTOCOL.md) ─────────
import sys, pathlib
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
from helper_functions.figures import setup_autosave
setup_autosave(study="TimeLapse_Cleaning", prefix="TLC_")


## Section 0 — Load Noisy Migrated Data & Physics Parameters

In [ ]:
v_ice = 0.168    # m/ns
f_c   = 1.5      # GHz
lam   = v_ice / f_c
kz_c  = 2 * np.pi / lam

STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\timelapse_study')
dn = np.load(str(STUDY_ROOT / 'migrated_results_noisy.npz'), allow_pickle=False)

noisy_kirchhoff         = dn['kirchhoff']            # (8, n_z, n_x)
noisy_gazdag            = dn['gazdag'] if 'gazdag' in dn.files else None  # absent in current file (Kirchhoff-only)
noisy_scenarios         = list(dn['scenarios'])      # ['Baseline', '2λ', ...]
noisy_separation_lambda = dn['separation_lambda']    # [0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]
noisy_x_traces          = dn['x_traces']
noisy_z_img             = dn['z_img']
noisy_x_s1              = dn['x_s1']
noisy_x_s2              = float(dn['x_s2'])
noisy_z_scatterer       = float(dn['z_scatterer'])
noisy_z_top             = float(dn['z_top'])
noisy_noise_level       = float(dn['noise_level'])

noisy_dz_mig = float(noisy_z_img[1]    - noisy_z_img[0])
noisy_dx_mig = float(noisy_x_traces[1] - noisy_x_traces[0])

print(f"Loaded: {STUDY_ROOT / 'migrated_results_noisy.npz'}")
print(f"  kirchhoff   : {noisy_kirchhoff.shape}")
print(f"  gazdag      : {None if noisy_gazdag is None else noisy_gazdag.shape}")
print(f"  scenarios   : {noisy_scenarios}")
print(f"  noise_level : {noisy_noise_level}")
print(f"\nlambda = {lam*1e3:.1f} mm,  kz_c = {kz_c:.2f} rad/m")
print(f"Grid spacing:  dz={noisy_dz_mig*1e3:.2f} mm,  dx={noisy_dx_mig*1e3:.2f} mm")
print(f"\nNOTE: Gazdag's noisy migration has a known structural artifact "
      f"(see TimeLapse_Processing.ipynb Section 2c) — most candidates below are "
      f"benchmarked against Kirchhoff only for that reason.")

## Section 1 — Baseline: `estimate_shift_2d` (magnitude-weighted L2 phase-plane fit)

Plain phase-plane fit, no cleaning — the method used in `TimeLapse_Processing.ipynb`
Section 2 (clean data) and Section 2c's PSR-gated fallback. Everything else in this
notebook is benchmarked against this.

In [ ]:
def estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, kz_pos_only=False, force_dz_zero=False):
    """
    Fit a 2D phase plane phi(kz,kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum.
    Returns (dz_est, dx_est, phi_0, XS, kz_ax, kx_ax).

    kz_pos_only=True   — restrict WLS to KZ > 0 (use with analytic-signal inputs).
    force_dz_zero=True — fit only (dx, phi_0); KZ column dropped and dz_est returned
                         as 0.0. Use when vertical movement is known to be absent.
    """
    Nz, Nx = base.shape

    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))

    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS       = base_fft * np.conj(mon_fft)

    w   = np.abs(XS)
    phi = np.angle(XS)

    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    if kz_pos_only:
        mask &= (KZ > 0)

    W = w[mask]
    if force_dz_zero:
        A = np.column_stack([KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, c[0], c[1], XS, kz_ax, kx_ax
    else:
        A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return c[0], c[1], c[2], XS, kz_ax, kx_ax

## Section 2 — Robust Apex-Finding (shared across all techniques below)

The crop-centering step matters more than the fit method (full investigation in
`TimeLapse_Processing.ipynb` Section 2b). Per-scenario localization (find the peak
of `|env_mon - env_base|` independently for each monitor scenario) is fragile at
small shifts: the genuine signal change is tiny, so noise-driven differences
elsewhere in the image can dominate the argmax.

**Fix:** median-stack the difference envelope across *all* monitor scenarios before
the argmax. The genuine signal bumps the difference at the same location in every
scenario; noise-driven false peaks are scenario-specific and don't reinforce across
the stack. One shared apex is found per migration method and reused for every
scenario's crop.

In [ ]:
def find_shared_apex(base_img, mon_imgs, x_traces, lam_val, search_lam=1.0, z_img=None, z_top=None):
    """
    Median-stack the difference envelope across multiple monitor images to find
    one robust apex location for the moving scatterer (see Section 2 markdown).

    Two stages: (1) a coarse pass takes the x-column where the median-stacked
    difference envelope `diff_consensus` peaks anywhere in z; (2) a fine pass
    refines within a +/-search_lam window around that column. The fine pass
    weights by `env_base * diff_consensus` (not env_base alone) so it snaps to
    a location that is *both* a strong real reflector *and* shows genuine
    cross-scenario difference — env_base alone can lock onto an unrelated bright
    reflector elsewhere in the window (confirmed against the regenerated
    noise_level=0.01 dataset, where a stronger nearby reflector pulled every
    estimator onto the wrong apex even though the coarse pass was correct).

    z_img/z_top: if both given, the air-gap region above z_top is excluded from
    the search (relevant for migration methods with near-surface artifacts —
    see TimeLapse_Processing.ipynb Section 2c re: Gazdag).
    """
    env_base = np.abs(sp_hilbert(np.nan_to_num(base_img), axis=0))
    diff_envs = [np.abs(np.abs(sp_hilbert(np.nan_to_num(m), axis=0)) - env_base) for m in mon_imgs]
    diff_consensus = np.median(np.stack(diff_envs, axis=0), axis=0)

    env_base_masked = env_base
    if z_img is not None and z_top is not None:
        z_mask = z_img >= z_top
        diff_consensus = diff_consensus.copy(); diff_consensus[~z_mask, :] = 0.0
        env_base_masked = env_base.copy(); env_base_masked[~z_mask, :] = 0.0

    _, ix_d = np.unravel_index(np.argmax(diff_consensus), diff_consensus.shape)
    x_rough = x_traces[ix_d]
    hw = search_lam * lam_val
    ix_lo = np.searchsorted(x_traces, x_rough - hw)
    ix_hi = np.searchsorted(x_traces, x_rough + hw)
    refine_score = env_base_masked[:, ix_lo:ix_hi] * diff_consensus[:, ix_lo:ix_hi]
    _, ix_loc = np.unravel_index(np.argmax(refine_score), refine_score.shape)
    return x_traces[ix_lo + ix_loc]


def crop_around(img, x_traces, x_apex, lam_val, half_width_lam=2.5):
    hw = half_width_lam * lam_val
    ix_lo = np.searchsorted(x_traces, x_apex - hw)
    ix_hi = np.searchsorted(x_traces, x_apex + hw)
    return img[:, ix_lo:ix_hi], x_traces[ix_lo:ix_hi]


# Shared Kirchhoff apex, reused throughout this notebook
_mon_imgs_kir = [noisy_kirchhoff[i] for i in range(1, len(noisy_scenarios))]
x_apex_kirchhoff = find_shared_apex(noisy_kirchhoff[0], _mon_imgs_kir, noisy_x_traces, lam)
print(f"Shared Kirchhoff apex: x={x_apex_kirchhoff:.4f} m  (true x_s1[0]={noisy_x_s1[0]:.4f} m)")

## Section 2b — How Many Scenarios Does the Stacking Fix Actually Need?

Section 2's fix relies on stacking the difference envelope across *all* monitor
scenarios — but a real time-lapse survey might only have a handful of monitor
acquisitions, not 7 known synthetic shifts. This stress-tests the fix: for every
possible subset of size K (K=1..7) of the 7 monitor scenarios, build the
median-stacked consensus apex from *only* that subset and check how close it
lands to the true scatterer position.

**Result:** K=1 (the original per-scenario method) succeeds only 57% of the time
(4 of 7 scenarios individually find the right apex; the other 3 — ⅛λ, ¹⁄₁₆λ,
¹⁄₃₂λ — fail). Going from K=1 to K=2 is the single biggest jump, to 90.5%. By
K=5 it's 100% reliable in this dataset (with only 3 structurally-weak scenarios
out of 7, any 5-scenario combination is guaranteed to include enough
strong-signal scenarios to outvote them).

**The K=1 failures are not random noise** — all three land on the *exact same*
wrong location (1348mm error, every time), not scattered values. That means it's
a fixed secondary peak in the *baseline* image's own noisy envelope (the baseline
is generated once and reused for every comparison) rather than per-scenario
randomness: whichever monitor scenario's rough search happens to land near that
baseline-side feature gets pulled onto it during refinement, regardless of which
scenario triggered the search.

**Practical implication:** a genuine one-shot single-pair survey (one baseline,
one monitor, ever) is fragile in a way this fix cannot help with — success would
depend on whether that particular baseline happens to have an unlucky secondary
peak, which can't be known in advance. But any real monitoring study with 2+
monitor surveys (the realistic case for "time-lapse") already gets most of the
benefit. A true single-pair fallback would need a different approach entirely —
e.g. a shape-aware matched filter against the expected scatterer PSF (integrates
evidence over the target's known footprint instead of trusting one pixel's
argmax), or geometry-constrained search if the approximate target location is
known from survey planning.

In [ ]:
import itertools

GOOD_THRESH_MM = 10.0
all_mon_idx = list(range(1, len(noisy_scenarios)))
all_mon_imgs = {i: noisy_kirchhoff[i] for i in all_mon_idx}
true_x = noisy_x_s1[0]

print(f"True apex x_s1[0] = {true_x:.4f} m\n")
print(f"{'K':>3s} {'n_combos':>9s} {'pct_good(<10mm)':>16s} {'best_err(mm)':>12s} {'worst_err(mm)':>13s}")

for K in range(1, len(all_mon_idx) + 1):
    combos = list(itertools.combinations(all_mon_idx, K))
    errs = []
    for combo in combos:
        mon_imgs = [all_mon_imgs[i] for i in combo]
        x_apex = find_shared_apex(noisy_kirchhoff[0], mon_imgs, noisy_x_traces, lam)
        errs.append(abs(x_apex - true_x) * 1e3)
    errs = np.array(errs)
    pct_good = 100.0 * np.mean(errs < GOOD_THRESH_MM)
    print(f"{K:3d} {len(combos):9d} {pct_good:16.1f} {errs.min():12.2f} {errs.max():13.2f}")

print("\n--- K=1 (per-scenario, the original method) breakdown ---")
for i in all_mon_idx:
    x_apex = find_shared_apex(noisy_kirchhoff[0], [all_mon_imgs[i]], noisy_x_traces, lam)
    err_mm = abs(x_apex - true_x) * 1e3
    print(f"  scenario {noisy_scenarios[i]:8s}: apex_err={err_mm:8.1f} mm")

## Section 3 — B-Scan-Domain Cleaning

Operates on the **full migrated image**, before cropping/apex-finding. Candidates
benchmarked in dev work:

- **Full-image median filter** — cheap baseline, no observed downside.
- **Wavelet denoising** (BayesShrink) — as configured, did essentially nothing
  (threshold too conservative); kept here for further tuning.
- **Multi-scenario low-rank/SVD** — stacks all 8 scenarios and truncates to a
  shared low-rank model. Did **not** work well: with only 8 samples, truncation
  destroys real shift signal along with noise (e.g. rank=2 turned a 224mm true
  shift into a 68mm estimate). Kept here in case it's worth revisiting with a
  different rank or a smarter signal/noise separation.
- **F-X deconvolution** — predictive filtering per depth-wavenumber slice across
  traces, exploiting that real reflectors are laterally predictable while random
  noise isn't. Best average performer in dev testing, but with one bad outlier
  (a scenario where it was worse than doing nothing) traced back to the apex-finding
  bug in Section 2, not the filter itself — worth re-benchmarking now that Section 2
  is fixed.

In [ ]:
def clean_median(stack, kernel_size=3):
    return np.stack([medfilt2d(img.astype(np.float64), kernel_size=kernel_size) for img in stack], axis=0)


def clean_wavelet(stack):
    return np.stack([
        denoise_wavelet(img, method='BayesShrink', mode='soft', rescale_sigma=True, channel_axis=None)
        for img in stack
    ], axis=0)


def clean_svd(stack, rank):
    """Stack all scenarios into an (n_scen, n_z*n_x) matrix and keep only the
    top `rank` singular components. Best used as an exploratory tool, not a
    default — see markdown above re: signal loss at low rank."""
    n_scen, Nz, Nx = stack.shape
    M = stack.reshape(n_scen, Nz * Nx)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    Mc = (U[:, :rank] * S[:rank]) @ Vt[:rank, :]
    return Mc.reshape(n_scen, Nz, Nx)


def clean_fx(stack, order=3):
    """F-X deconvolution: FFT along depth (z) -> per-kz-row complex AR(order)
    prediction filter across x -> IFFT back to the space domain. A single
    global filter per row (not a sliding window) — simplification worth
    revisiting if results are promising."""
    out = []
    for img in stack:
        Nz, Nx = img.shape
        D = np.fft.fft(img, axis=0)
        D_clean = np.empty_like(D)
        for iz in range(Nz):
            d = D[iz, :]
            if Nx <= 2 * order:
                D_clean[iz, :] = d
                continue
            A = np.column_stack([d[order - 1 - k: Nx - 1 - k] for k in range(order)])
            y = d[order:]
            coef, *_ = np.linalg.lstsq(A, y, rcond=None)
            pred = A @ coef
            d_out = d.copy()
            d_out[order:] = pred
            D_clean[iz, :] = d_out
        out.append(np.real(np.fft.ifft(D_clean, axis=0)))
    return np.stack(out, axis=0)

## Section 4 — Cross-Spectrum-Domain Cleaning

Operates on the 2-D FFT cross-spectrum of the (already cropped) base/monitor pair,
before the shift estimate is extracted from it.

**Note on the GCC sign convention:** `XS = base_fft * conj(mon_fft)` gives
`angle(XS) = k*dx` (the convention `estimate_shift_2d` uses), but numpy's `ifft`
kernel puts the raw correlation peak at `lag = -dx`. The lag axes in `estimate_shift_2d_gcc`
below are pre-negated to correct for this — verified analytically and against a
synthetic shifted signal during dev work.

In [ ]:
def estimate_shift_2d_huber(base, mon, dz_g, dx_g, kz_cent, epsilon=1.35):
    """Same cross-spectrum setup as estimate_shift_2d, but fits the (kz,kx) phase
    plane with Huber regression instead of weighted lstsq. In dev testing this
    showed no consistent advantage over plain lstsq at this crop size."""
    Nz, Nx = base.shape
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS = base_fft * np.conj(mon_fft)
    w, phi = np.abs(XS), np.angle(XS)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    A = np.column_stack([KZ[mask], KX[mask]])
    huber = HuberRegressor(epsilon=epsilon)
    huber.fit(A, phi[mask], sample_weight=w[mask])
    return huber.coef_[0], huber.coef_[1], huber.intercept_, XS, kz_ax, kx_ax


def estimate_shift_2d_ransac(base, mon, dz_g, dx_g, kz_cent, mask_frac=0.10, random_state=0):
    """Same cross-spectrum setup, RANSAC plane fit instead of weighted lstsq.
    In dev testing this was a clear win for 1/4lambda-1lambda shifts (explicitly
    discards outlier bins rather than downweighting them) but didn't fix the
    smallest shifts."""
    Nz, Nx = base.shape
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS = base_fft * np.conj(mon_fft)
    w, phi = np.abs(XS), np.angle(XS)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > mask_frac * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    A = np.column_stack([KZ[mask], KX[mask]])
    ransac = RANSACRegressor(estimator=LinearRegression(), random_state=random_state)
    ransac.fit(A, phi[mask], sample_weight=w[mask])
    coef = ransac.estimator_.coef_
    return coef[0], coef[1], 0.0, XS, kz_ax, kx_ax


def estimate_shift_2d_gcc(base, mon, dz_g, dx_g, kz_cent,
                           coarse_search_lam=3.0, fine_search_lam=0.5):
    """Generalized cross-correlation: correlate base/monitor directly in the
    space domain (R = real(ifft2(XS))) and locate the shift as the peak of R,
    instead of fitting a plane to phase. Most accurate candidate in dev testing
    for shifts >~ 1/4 wavelength; unreliable below that (see
    estimate_shift_2d_gcc_psr for the production version with a confidence-gated
    fallback, used in TimeLapse_Processing.ipynb Section 2c).

    Returns (dz_est, dx_est, psr, XS, kz_ax, kx_ax) — `psr` is the peak-to-sidelobe
    ratio of the correlation envelope (~10 = trustworthy, ~4.5-6 = not, in dev
    testing); use it to decide whether to trust this estimate or fall back to
    something else.
    """
    Nz, Nx = base.shape
    lam_val = 2 * np.pi / kz_cent
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS = base_fft * np.conj(mon_fft)
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi

    R = np.real(np.fft.ifft2(XS))
    R_shift = np.fft.fftshift(R)
    # sign fix: numpy's ifft kernel puts the peak at lag = -true_shift
    lag_z = -(np.arange(Nz) - Nz // 2) * dz_g
    lag_x = -(np.arange(Nx) - Nx // 2) * dx_g
    oz, ox = np.argsort(lag_z), np.argsort(lag_x)
    lag_z, lag_x = lag_z[oz], lag_x[ox]
    R_shift = R_shift[np.ix_(oz, ox)]

    env = np.abs(sp_hilbert(R_shift, axis=1))
    hw_c = coarse_search_lam * lam_val
    zmask_c = np.abs(lag_z) <= hw_c
    xmask_c = np.abs(lag_x) <= hw_c
    sub_env = env[np.ix_(zmask_c, xmask_c)]
    iz_c, ix_c = np.unravel_index(np.argmax(sub_env), sub_env.shape)
    lag_z_c, lag_x_c = lag_z[zmask_c][iz_c], lag_x[xmask_c][ix_c]
    peak_env = sub_env[iz_c, ix_c]
    background = np.delete(sub_env.ravel(), np.argmax(sub_env.ravel()))
    psr = (peak_env - background.mean()) / (background.std() + 1e-12)

    hw_f = fine_search_lam * lam_val
    zmask_f = np.abs(lag_z - lag_z_c) <= hw_f
    xmask_f = np.abs(lag_x - lag_x_c) <= hw_f
    sub_R = R_shift[np.ix_(zmask_f, xmask_f)]
    lag_z_f, lag_x_f = lag_z[zmask_f], lag_x[xmask_f]
    iz0, ix0 = np.unravel_index(np.argmax(np.abs(sub_R)), sub_R.shape)

    def _parabolic(vals, idx, axis_vals):
        if idx <= 0 or idx >= len(axis_vals) - 1:
            return axis_vals[idx]
        y0, y1, y2 = vals[idx - 1], vals[idx], vals[idx + 1]
        denom = y0 - 2 * y1 + y2
        if denom == 0:
            return axis_vals[idx]
        delta = 0.5 * (y0 - y2) / denom
        d = axis_vals[1] - axis_vals[0]
        return axis_vals[idx] + delta * d

    dz_est = _parabolic(sub_R[:, ix0], iz0, lag_z_f)
    dx_est = _parabolic(sub_R[iz0, :], ix0, lag_x_f)
    return dz_est, dx_est, psr, XS, kz_ax, kx_ax


def estimate_shift_2d_gcc_psr(base, mon, dz_g, dx_g, kz_cent, medfilt_size=3, psr_threshold=8.0):
    """Production version (matches estimate_shift_2d_cleaning in
    TimeLapse_Processing.ipynb): median-filter the crops, run GCC, and fall back
    to estimate_shift_2d's lstsq fit when the GCC peak's PSR is below threshold.
    Magnitude alone can't gate the fallback (failed small-shift estimates can
    coincidentally match the magnitude of genuine large-shift ones); PSR can.
    """
    base_clean = medfilt2d(base.astype(np.float64), kernel_size=medfilt_size)
    mon_clean  = medfilt2d(mon.astype(np.float64),  kernel_size=medfilt_size)
    dz_gcc, dx_gcc, psr, XS, kz_ax, kx_ax = estimate_shift_2d_gcc(base_clean, mon_clean, dz_g, dx_g, kz_cent)
    if psr < psr_threshold:
        dz_est, dx_est, phi_0, _, _, _ = estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent)
        return dz_est, dx_est, phi_0, XS, kz_ax, kx_ax
    return dz_gcc, dx_gcc, 0.0, XS, kz_ax, kx_ax

## Section 5 — Benchmark: Compare Candidates Against True Δx

Mix and match: pick a B-scan cleaning function from Section 3 (or `None`) and a
cross-spectrum estimator from Section 1/4, run it across all 7 Kirchhoff scenarios
using the shared apex from Section 2, and compare to the true shift.

In [ ]:
def benchmark(bscan_clean_fn, estimator_fn, label, base_stack=noisy_kirchhoff):
    stack = base_stack if bscan_clean_fn is None else bscan_clean_fn(base_stack)
    base_img = stack[0]
    mon_imgs = [stack[i] for i in range(1, len(noisy_scenarios))]
    x_apex = find_shared_apex(base_img, mon_imgs, noisy_x_traces, lam)
    base_crop, x_crop = crop_around(base_img, noisy_x_traces, x_apex, lam)

    print(f"\n--- {label}  (apex @ x={x_apex:.4f} m) ---")
    print(f"{'scenario':8s} {'true(mm)':>9s} {'est(mm)':>9s} {'err(mm)':>9s}")
    for i in range(1, len(noisy_scenarios)):
        true_dx = noisy_separation_lambda[i] * lam
        mon_crop, _ = crop_around(stack[i], noisy_x_traces, x_apex, lam)
        dz_est, dx_est, *_ = estimator_fn(base_crop, mon_crop, noisy_dz_mig, noisy_dx_mig, kz_c)
        err = dx_est - true_dx
        print(f"{noisy_scenarios[i]:8s} {true_dx*1e3:9.2f} {dx_est*1e3:9.2f} {err*1e3:9.2f}")


# ── Example comparisons — edit/extend freely ──────────────────────────────────
benchmark(None,            estimate_shift_2d,        "baseline lstsq, no cleaning")
benchmark(None,            estimate_shift_2d_gcc_psr, "GCC + PSR fallback, no B-scan cleaning")
benchmark(clean_median,    estimate_shift_2d_gcc_psr, "GCC + PSR fallback, full-image median filter")
benchmark(clean_fx,        estimate_shift_2d_gcc_psr, "GCC + PSR fallback, F-X deconvolution")
benchmark(None,            estimate_shift_2d_ransac,  "RANSAC plane fit, no B-scan cleaning")
benchmark(None,            estimate_shift_2d_huber,   "Huber plane fit, no B-scan cleaning")

## Section 6 — Phase 1 + Phase 2: Crop-Level Median Filter + Cross-Spectrum Smoothing + Huber Fit

Two cleaning stages, combined here for the first time in this notebook:

- **Phase 1 (B-scan domain, right before the FFT):** median-filter `base_crop`/`mon_crop`
  themselves — not the full image (Section 3's `clean_median`, applied before
  cropping) and not as an isolated step (Section 4's `estimate_shift_2d_gcc_psr`
  already does this, but only for the GCC estimator — the lstsq/Huber plane-fit
  estimators in Sections 1/4 never get a cleaned crop). Laplace noise is impulsive
  ("salt-and-pepper"-like), so a 3×3/5×5 median is a good match: it erases isolated
  spikes and migration-smile arcs while leaving the sharp front boundary intact.
- **Phase 2 (cross-spectrum domain, right before `np.angle`):** smooth the *complex*
  cross-spectrum `XS = base_fft * conj(mon_fft)` with a small uniform boxcar
  (real and imaginary parts filtered separately, then recombined) so random
  per-pixel noise phases cancel destructively *before* being unwrapped into an
  angle, rather than surviving into the phase map as ±180° outliers. Then fit the
  (kz, kx) phase plane with `HuberRegressor` instead of weighted lstsq, so any
  phase pixels the boxcar didn't fully clean up get downweighted instead of
  squared.

Section 4 tried Huber alone (no boxcar smoothing) and found "no consistent
advantage over plain lstsq" — the hypothesis here is that Huber needs a
pre-smoothed phase map to do useful work: with raw per-pixel phase noise spanning
the full ±180° range, *most* low-SNR bins are corrupted, not just a few isolated
ones, so downweighting outliers alone can't fix it. The boxcar should turn that
into a smaller number of genuine outliers that Huber can then handle.

**Unrelated problem found while testing this on the current noisy dataset:** the
data file was regenerated with `noise_level=0.01` (was `0.1`), and at that lower
noise every estimator in this notebook — including the previously-validated
baseline and GCC ones — returns ≈0.00mm for every scenario. This is *not* the
impulsive-noise problem Phase 1/2 target. Root cause: the crop contains a strong
**laterally-invariant background** (the air/ice interface reflection — identical
at every x-column in both base and monitor images), which carries ~100x more
spectral energy than the moving scatterer's diffraction signature but sits
entirely at kx=0, i.e. carries zero lateral-shift information. It swamps the
magnitude-weighted threshold mask (every method here selects almost exclusively
kx=0 bins) and dominates GCC's real-space correlation peak (a coherent, identical
signal correlates trivially at zero lag). Median filtering and boxcar smoothing
don't fix this because it isn't random — it's coherent, structural energy.

**Fix (Section 6a below, applied before Phase 1):** subtract each crop's own
per-depth (per-z) mean across x from itself. The interface reflection is constant
across x within the crop, so it lives entirely in that per-row mean; subtracting
it removes the dominant kx=0 component and leaves the x-varying point-scatterer
signature untouched. Confirmed standalone: this alone took GCC from 0.00mm on
every scenario to within ~4-14mm of the true shift across all seven (limited by
the 10mm trace grid at the smallest shifts) — before Phase 1/2 are even applied.

One function below supports five independently-toggleable stages (background
removal, median filter, XS smoothing, plus a choice of lstsq vs. Huber for the
final fit), so the ablation can isolate which combination actually matters on
this dataset.

In [ ]:
def estimate_shift_2d_cropmed_smooth(base, mon, dz_g, dx_g, kz_cent,
                                      remove_bg=True, medfilt_size=3, smooth_size=3,
                                      fit='huber', epsilon=1.35):
    """
    Stage 0 (background): subtract each crop's own per-depth mean across x —
    removes the laterally-invariant interface reflection that otherwise swamps
    the kx=0 bin (remove_bg=False disables this stage).
    Phase 1: median-filter the (already-cropped) base/mon crops right before the
    2D FFT (medfilt_size=None disables this stage).
    Phase 2: boxcar-smooth the complex cross-spectrum before np.angle()
    (smooth_size=1 disables this stage), then fit the (kz,kx) phase plane with
    weighted lstsq (fit='lstsq') or Huber regression (fit='huber').
    """
    if remove_bg:
        base = base - base.mean(axis=1, keepdims=True)
        mon  = mon  - mon.mean(axis=1, keepdims=True)

    if medfilt_size is not None:
        base = medfilt2d(base.astype(np.float64), kernel_size=medfilt_size)
        mon  = medfilt2d(mon.astype(np.float64),  kernel_size=medfilt_size)

    Nz, Nx = base.shape
    kz_ax = np.fft.fftfreq(Nz, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')

    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    base_fft = np.fft.fft2(base * taper)
    mon_fft  = np.fft.fft2(mon  * taper)
    XS = base_fft * np.conj(mon_fft)

    if smooth_size > 1:
        XS = uniform_filter(XS.real, size=smooth_size) + 1j * uniform_filter(XS.imag, size=smooth_size)

    w, phi = np.abs(XS), np.angle(XS)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    if fit == 'huber':
        A = np.column_stack([KZ[mask], KX[mask]])
        huber = HuberRegressor(epsilon=epsilon)
        huber.fit(A, phi[mask], sample_weight=w[mask])
        return huber.coef_[0], huber.coef_[1], huber.intercept_, XS, kz_ax, kx_ax
    else:
        W = w[mask]
        A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return c[0], c[1], c[2], XS, kz_ax, kx_ax

In [ ]:
def estimate_shift_2d_gcc_bg(base, mon, dz_g, dx_g, kz_cent, medfilt_size=3, psr_threshold=8.0):
    """estimate_shift_2d_gcc_psr (Section 4) with the same background-removal
    stage prepended, for comparison — GCC's real-space correlation peak is just
    as vulnerable to the laterally-invariant background as the phase-plane fits
    (a coherent, identical signal correlates trivially at zero lag)."""
    base_bg = base - base.mean(axis=1, keepdims=True)
    mon_bg  = mon  - mon.mean(axis=1, keepdims=True)
    return estimate_shift_2d_gcc_psr(base_bg, mon_bg, dz_g, dx_g, kz_cent,
                                      medfilt_size=medfilt_size, psr_threshold=psr_threshold)

In [ ]:
from functools import partial

variants = [
    ("no cleaning at all (lstsq)",                          partial(estimate_shift_2d_cropmed_smooth, remove_bg=False, medfilt_size=None, smooth_size=1, fit='lstsq')),
    ("bg removal only (lstsq)",                             partial(estimate_shift_2d_cropmed_smooth, remove_bg=True,  medfilt_size=None, smooth_size=1, fit='lstsq')),
    ("bg removal + crop median 3x3 (lstsq)",                partial(estimate_shift_2d_cropmed_smooth, remove_bg=True,  medfilt_size=3,    smooth_size=1, fit='lstsq')),
    ("bg removal + crop median 3x3 + XS boxcar 3x3 (lstsq)", partial(estimate_shift_2d_cropmed_smooth, remove_bg=True,  medfilt_size=3,    smooth_size=3, fit='lstsq')),
    ("bg removal + crop median 3x3 + XS boxcar 3x3 (huber)", partial(estimate_shift_2d_cropmed_smooth, remove_bg=True,  medfilt_size=3,    smooth_size=3, fit='huber')),
    ("bg removal + crop median 5x5 + XS boxcar 5x5 (huber)", partial(estimate_shift_2d_cropmed_smooth, remove_bg=True,  medfilt_size=5,    smooth_size=5, fit='huber')),
]

for label, fn in variants:
    benchmark(None, fn, label)

# context: GCC alone (Section 4) vs. GCC + the same background-removal fix
benchmark(None, estimate_shift_2d_gcc_psr, "GCC + PSR fallback, no bg removal (Section 4, for reference)")
benchmark(None, estimate_shift_2d_gcc_bg,  "GCC + PSR fallback, WITH bg removal")

### Section 6 — Findings

On the current dataset (`noise_level=0.01`), the requested Phase 1/Phase 2 steps
**do not help, and slightly hurt**:

- **Background removal is the real fix.** It alone takes every scenario from
  0.00mm (completely broken) to sub-mm/few-mm accuracy for ½λ down to ¹⁄₃₂λ.
  It does *not* fix the lstsq/Huber plane fit at 2λ/1λ — that's the
  already-documented phase-wrapping failure mode (Section 5: a 2λ lateral shift
  wraps the phase ~2 full cycles at the band edge, which a per-pixel angle-based
  plane fit cannot resolve without unwrapping).
- **Crop median filtering (Phase 1) and XS boxcar smoothing + Huber (Phase 2),
  layered on top of background removal, monotonically worsen the small/mid-shift
  estimates** (e.g. ⅛λ error grows from -0.98mm with bg-removal alone to -1.74mm
  with median+boxcar+Huber, to -3.83mm with 5×5 kernels). At `noise_level=0.01`
  there isn't enough impulsive noise left for median filtering to have anything
  useful to remove, and both the median filter and the boxcar smoothing blur
  real structure they didn't need to blur — a real cost with no offsetting
  benefit at this noise level.
- **Best result in this notebook:** background removal + Section 4's existing
  `estimate_shift_2d_gcc_psr` (`estimate_shift_2d_gcc_bg` above). Errors across
  all seven scenarios: -6.63, 0.91, 0.07, -0.19, -0.31, 0.07, 0.42mm — including
  the two large shifts (2λ, 1λ) that every phase-plane-fit variant fails on,
  because GCC estimates the shift from the correlation peak's *location* in
  real space rather than fitting wrapped phase angles, so it has no wrapping
  failure mode to begin with.

**Practical takeaway:** the impulsive-noise-targeted cleaning requested at the
top of this section is the right idea for a *noisier* dataset (it was a clear
to call at `noise_level=0.1` in earlier sections), but isn't the binding
constraint here — background removal is. If the real survey data this is meant
to emulate has noise levels closer to 0.1 than 0.01, Phase 1/2 are worth
revisiting on that regime specifically (ideally combined with background
removal, which costs nothing and never hurt any candidate above).

### Section 6b — Visualizing Background Removal + GCC

For a few representative scenarios: the raw crop pair, the same crop pair after
background removal (per-depth mean subtracted), and the resulting GCC
cross-correlation — a 2D heatmap zoomed near the peak, plus a 1D slice through
`lag_z=0` showing exactly where the peak lands relative to the true shift.

In [ ]:
def plot_bg_gcc(scenario_idx, medfilt_size=3, zoom_lam=1.5):
    base_img = noisy_kirchhoff[0]
    mon_img  = noisy_kirchhoff[scenario_idx]
    base_crop, x_crop = crop_around(base_img, noisy_x_traces, x_apex_kirchhoff, lam)
    mon_crop, _        = crop_around(mon_img,  noisy_x_traces, x_apex_kirchhoff, lam)
    x_crop_mm = x_crop * 1e3
    z_mm = noisy_z_img * 1e3

    base_bg = base_crop - base_crop.mean(axis=1, keepdims=True)
    mon_bg  = mon_crop  - mon_crop.mean(axis=1, keepdims=True)

    base_clean = medfilt2d(base_bg.astype(np.float64), kernel_size=medfilt_size)
    mon_clean  = medfilt2d(mon_bg.astype(np.float64),  kernel_size=medfilt_size)

    dz_est, dx_est, psr, XS, kz_ax, kx_ax = estimate_shift_2d_gcc(
        base_clean, mon_clean, noisy_dz_mig, noisy_dx_mig, kz_c)

    Nz, Nx = base_crop.shape
    R = np.real(np.fft.ifft2(XS))
    R_shift = np.fft.fftshift(R)
    lag_z = -(np.arange(Nz) - Nz // 2) * noisy_dz_mig
    lag_x = -(np.arange(Nx) - Nx // 2) * noisy_dx_mig
    oz, ox = np.argsort(lag_z), np.argsort(lag_x)
    lag_z, lag_x = lag_z[oz], lag_x[ox]
    R_shift = R_shift[np.ix_(oz, ox)]
    env = np.abs(sp_hilbert(R_shift, axis=1))

    true_dx = noisy_separation_lambda[scenario_idx] * lam
    label = noisy_scenarios[scenario_idx]

    # zoom window must cover both true_dx and dx_est, not just a fixed wavelength multiple
    hw = max(zoom_lam * lam, 1.2 * abs(true_dx), 1.2 * abs(dx_est), lag_x.max())
    hw = min(hw, lag_x.max())
    zmask = np.abs(lag_z) <= zoom_lam * lam
    xmask = np.abs(lag_x) <= hw
    iz0 = np.argmin(np.abs(lag_z))  # row closest to lag_z=0

    vmax_raw = np.percentile(np.abs(base_crop), 99)
    vmax_bg  = np.percentile(np.abs(base_bg), 99)

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f"Background removal + GCC — scenario {label} "
                 f"(true dx={true_dx*1e3:.1f} mm, est={dx_est*1e3:.1f} mm)")

    extent_img = [x_crop_mm[0], x_crop_mm[-1], z_mm[-1], z_mm[0]]
    axes[0, 0].imshow(base_crop, extent=extent_img, aspect='auto', cmap='seismic',
                       vmin=-vmax_raw, vmax=vmax_raw)
    axes[0, 0].set_title('base_crop (raw)')
    axes[0, 0].set_xlabel('x (mm)'); axes[0, 0].set_ylabel('z (mm)')

    axes[0, 1].imshow(mon_crop, extent=extent_img, aspect='auto', cmap='seismic',
                       vmin=-vmax_raw, vmax=vmax_raw)
    axes[0, 1].set_title(f'mon_crop (raw), {label}')
    axes[0, 1].set_xlabel('x (mm)'); axes[0, 1].set_ylabel('z (mm)')

    sub_env = env[np.ix_(zmask, xmask)]
    im = axes[0, 2].imshow(sub_env, extent=[lag_x[xmask][0]*1e3, lag_x[xmask][-1]*1e3,
                                             lag_z[zmask][-1]*1e3, lag_z[zmask][0]*1e3],
                            aspect='auto', cmap='viridis')
    axes[0, 2].axvline(true_dx*1e3, color='lime', ls='--', lw=1.5, label='true dx')
    axes[0, 2].axvline(dx_est*1e3, color='red', ls='-', lw=1.5, label='estimated dx')
    axes[0, 2].set_title('GCC correlation envelope (zoomed)')
    axes[0, 2].set_xlabel('lag_x (mm)'); axes[0, 2].set_ylabel('lag_z (mm)')
    axes[0, 2].legend(loc='upper right', fontsize=8)
    plt.colorbar(im, ax=axes[0, 2], fraction=0.046)

    axes[1, 0].imshow(base_bg, extent=extent_img, aspect='auto', cmap='seismic',
                       vmin=-vmax_bg, vmax=vmax_bg)
    axes[1, 0].set_title('base_crop (bg removed)')
    axes[1, 0].set_xlabel('x (mm)'); axes[1, 0].set_ylabel('z (mm)')

    axes[1, 1].imshow(mon_bg, extent=extent_img, aspect='auto', cmap='seismic',
                       vmin=-vmax_bg, vmax=vmax_bg)
    axes[1, 1].set_title('mon_crop (bg removed)')
    axes[1, 1].set_xlabel('x (mm)'); axes[1, 1].set_ylabel('z (mm)')

    profile = env[iz0, :]
    axes[1, 2].plot(lag_x * 1e3, profile, color='black', lw=1)
    axes[1, 2].set_xlim(-hw*1e3, hw*1e3)
    axes[1, 2].axvline(true_dx*1e3, color='lime', ls='--', lw=1.5, label='true dx')
    axes[1, 2].axvline(dx_est*1e3, color='red', ls='-', lw=1.5, label='estimated dx')
    axes[1, 2].set_title(f'correlation envelope @ lag_z≈0  (PSR={psr:.1f})')
    axes[1, 2].set_xlabel('lag_x (mm)'); axes[1, 2].set_ylabel('|envelope|')
    axes[1, 2].legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()


for idx in [1, 4, 7]:  # 2λ (large), ¼λ (mid), ¹⁄₃₂λ (smallest)
    plot_bg_gcc(idx)